In [ ]:
from pathlib import Path
from typing import Literal
import string
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch

import cartopy.crs as ccrs
import cartopy.feature as cfeature
from common import load_data, ggcms, ssps

from matplotlib.colors import LinearSegmentedColormap
import matplotlib as mpl

In [ ]:
def format_feature_name(name: str) -> str:
    return {
        "TMXav": "Δ Max. temperature [°C/day]", 
        "PRCPsum": "Δ Precipitation [mm/day]", 
        "RADsum": "Δ Radiation [MJ/m²/day]", 
        "KDD": "Δ Killing degree days [% of GS]", 
        "PETsum": "Δ PET [mm/day]",
        "CMDgt0": "Δ Days where CMD > 0 [% of GS]"
    }[name]

In [ ]:
features_plot = ["TMXav", "PRCPsum", "RADsum", "KDD", "CMDgt0", "PETsum"]
ggcms_plot = ["MPI-ESM1-2-HR", "UKESM1-0-LL"]

In [ ]:
labels = string.ascii_lowercase
ssp = "ssp370"

hist_from = 1985
hist_to = 2014

year_from = 2070
year_to = 2099

mpl.colormaps.unregister("PinkSky")

pinksky = LinearSegmentedColormap.from_list(
    "PinkSky",
    [
        "#4fc3f7",   # bright sky blue
        "#ffffff",
        "#d81b60",   # magenta
    ],
)

mpl.colormaps.register(pinksky)

cmaps = {
    "TMXav": "Reds",
    "PRCPsum": "BrBG",
    "PETsum": "PuOr",
    "CMDgt0": "RdBu",
    "KDD": "YlOrRd",
    "RADsum": "PinkSky"
}

plt.rcParams.update({"font.size": 13})

fig, axs = plt.subplots(
    len(features_plot),
    len(ggcms_plot),
    figsize=(15, 20),
    subplot_kw={"projection": ccrs.Robinson()},
    layout="compressed",
)

for row, feature in enumerate(features_plot):

    deltas = []

    ###########################################################
    # Load both GCMs
    ###########################################################

    for ggcm in ggcms_plot:

        hist = (
            load_data("historical", ggcm, "mai")
            .filter(
                pl.col("YR").is_between(hist_from, hist_to),
                pl.col("PERIOD") == "gs",
            )
            .group_by(["LAT", "LON"])
            .mean()
        )

        future = (
            load_data(ssp, ggcm, "mai")
            .filter(
                pl.col("YR").is_between(year_from, year_to),
                pl.col("PERIOD") == "gs",
            )
            .group_by(["LAT", "LON"])
            .mean()
        )

        # Normalize accumulated variables
        if feature in ["PRCPsum", "RADsum", "PETsum"]:
    
            hist = hist.with_columns(
                (pl.col(feature) / pl.col("LEN")).alias(feature)
            )
    
            future = future.with_columns(
                (pl.col(feature) / pl.col("LEN")).alias(feature)
            )
    
        elif feature in ["KDD", "CMDgt0"]:
    
            hist = hist.with_columns(
                (pl.col(feature) / pl.col("LEN") * 100).alias(feature)
            )
    
            future = future.with_columns(
                (pl.col(feature) / pl.col("LEN") * 100).alias(feature)
            )
    
        # Compute pixel-wise deltas
        delta = (
            future.join(
                hist.select(["LAT", "LON", feature]),
                on=["LAT", "LON"],
                suffix="_hist",
            )
            .with_columns(
                (pl.col(feature) - pl.col(f"{feature}_hist")).alias(feature)
            )
            .select(["LAT", "LON", feature])
        )
    
        deltas.append(delta)

    ###########################################################
    # Color normalization
    ###########################################################

    vals = np.concatenate([
        d[feature].to_numpy()
        for d in deltas
    ])

    if feature in ["TMXav", "KDD"]:

        vmax = np.percentile(vals, 99)

        norm = mcolors.Normalize(
            vmin=0,
            vmax=vmax,
        )

    elif feature in ["PRCPsum", "PETsum", "RADsum", "CMDgt0"]:

        vmax = np.percentile(np.abs(vals), 98)

        norm = mcolors.TwoSlopeNorm(
            vmin=-vmax,
            vcenter=0,
            vmax=vmax,
        )

    else:
        norm = None

    ###########################################################
    # Plot
    ###########################################################

    for col, (ggcm, delta) in enumerate(zip(ggcms_plot, deltas)):

        ax = axs[row, col]

        sc = ax.scatter(
            delta["LON"].to_numpy(),
            delta["LAT"].to_numpy(),
            c=delta[feature].to_numpy(),
            s=2,
            cmap=cmaps[feature],
            norm=norm,
            transform=ccrs.PlateCarree(),
            rasterized=True,
        )

        #ax.set_global()
        ax.set_extent(
            [-180, 180, -65, 65],
            crs=ccrs.PlateCarree(),
        )
        
        ax.add_feature(
            cfeature.LAND,
            facecolor="0.96",
            zorder=0,
        )

        ax.add_feature(
            cfeature.COASTLINE,
            linewidth=0.4,
        )

        ax.add_feature(
            cfeature.BORDERS,
            linewidth=0.2,
            alpha=0.3,
        )

        if row == 0:
            ax.set_title(ggcm)

        # Panel label
        panel = row * len(ggcms_plot) + col
        ax.text(
            0.02,
            0.95,
            labels[panel],
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=16,
            fontweight="bold",
        )

    ###########################################################
    # Colorbar
    ###########################################################

    cbar = fig.colorbar(
        sc,
        ax=axs[row, :],
        orientation="vertical",
        shrink=0.8,
        fraction=0.025,
        pad=0.02,
    )

    cbar.set_label(format_feature_name(feature))

fig.savefig(
    "features_maps.png",
    dpi=300,
    bbox_inches="tight",
    pad_inches=0.02,
)

plt.show()